<a href="https://colab.research.google.com/github/Innovatewithapple/RAGPractice/blob/main/TaleStoryRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import random
import torch

def setProductionSeed(isActive,seed=42):
  if isActive == True:
   random.seed(seed)
   os.environ['PYTHONHASHSEED'] = str(seed)
   np.random.seed(seed)

   torch.manual_seed(seed)

   if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 💥 THE EXPERT TRICK: Force CUDA core algorithms to stay 100% rigid [prev]
    # This stops the GPU from switching algorithms to save time, ensuring absolute repeatability
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
setProductionSeed(isActive=True,seed=42)

In [10]:
!pip install -q pypdf

In [11]:
import pypdf

In [12]:
def extract_and_chunkpdf(pdfPath,chunksize,overlap):
  print(f'Opening pdf from path: {pdfPath}')

  reader = pypdf.PdfReader(stream=pdfPath)

  #unpack text sequentially page by page
  full_text_list = []

  for page in reader.pages[1:]:
    text = page.extract_text()
    if text: # System check: ignore if there is any image or graphics
      full_text_list.append(text)

  full_raw_string = " ".join(full_text_list)

  #Run the sliding window splitter loop
  words = full_raw_string.split()
  chunks = []

  for i in range(0,len(words),chunksize-overlap):
    chunk_words = words[i:i + chunksize]
    paragraph_string = " ".join(chunk_words)
    chunks.append(paragraph_string)

  return chunks

In [13]:
knowledge_strings = extract_and_chunkpdf(pdfPath='/content/Tell-Tale_Heart.pdf',chunksize=150,overlap=30)
print(f'total overlapped chunks: {len(knowledge_strings)}')
print("\n--- SAMPLE VIEW OF INDEX 0 PARAGRAPH ---")
print(knowledge_strings[0][:400] + "...") # Preview the first 400 characters

Opening pdf from path: /content/Tell-Tale_Heart.pdf
total overlapped chunks: 20

--- SAMPLE VIEW OF INDEX 0 PARAGRAPH ---
COPYRIGHT INFORMATION Short Story: “The Tell-Tale Heart” Author: Edgar Allan Poe, 1809–49 First published: 1843 The original short story is in the public domain in the United States and in most, if not all, other countries as well. Readers outside the United States s hould check their own countries’ copyright l aws to b e ce rtain they can legally download this e-story. The Online Books Page has a...


In [14]:
!pip install -q datasets transformers

In [33]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset
from transformers import AutoModel,AutoTokenizer,AutoModelForCausalLM
from tqdm import tqdm
import torch.nn.functional as F

In [20]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

## **Encoder**

In [ ]:
encoder_autoToken = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

In [50]:
max_len = 256

In [51]:
encoder_token = encoder_autoToken(text=knowledge_strings,padding='max_length',max_length=max_len,add_special_tokens=True,return_attention_mask=True,return_tensors='pt')

In [52]:
class RAGKnowledgeDataset(Dataset):
  def __init__(self,encoding):
    self.encoding = encoding

  def __len__(self):
    return len(self.encoding['input_ids'])

  def __getitem__(self,idx):
    input_ids = self.encoding['input_ids'][idx]
    attention_mask = self.encoding['attention_mask'][idx]

    return {
        'input_ids':input_ids,
        'attention_mask':attention_mask
    }

In [53]:
encoded_Data = RAGKnowledgeDataset(encoder_token)

In [54]:
encodedData_Loader = DataLoader(dataset=encoded_Data,batch_size=16,shuffle=False,pin_memory=True,num_workers=2)

In [55]:
class BERTWriter(nn.Module):
  def __init__(self):
    super().__init__()
    self.bert = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

  def forward(self,input_ids,attention_mask):
    output = self.bert(input_ids=input_ids,attention_mask=attention_mask)
    x = output.last_hidden_state

    mean = attention_mask.unsqueeze(-1).float()
    mean = (x*mean).sum(dim=1) / mean.sum(dim=1)
    return mean

In [ ]:
encoder_Model = BERTWriter().to(device)
encoder_Model.eval()

In [57]:
database_vectors = []

with torch.no_grad():
  with torch.amp.autocast('cuda'):

    for batch in tqdm(encodedData_Loader,desc=f'Encoding Batchs'):
      input_ids,attention_mask = batch['input_ids'].to(device) , batch['attention_mask'].to(device)

      encoder_model_output = encoder_Model(input_ids,attention_mask)

      database_vectors.append(encoder_model_output.cpu()) # move it to cpu so gpu rams are clean

vector_Database = torch.cat(database_vectors,dim=0)
print(f'vector database shape: {vector_Database.shape}')

Encoding Batchs: 100%|██████████| 2/2 [00:00<00:00, 10.26it/s]

vector database shape: torch.Size([20, 384])


In [69]:
questions = [
    "Who did the narrator kill?",
    "Why did he hate the old man? Why did he decide to take his life or kill him?",
    "What made the narrator nervous?",
    "What happened to the body?",
    "Why did the narrator confess?",
    "What sound did he keep hearing?",
    "Where did he hide the body?"
]

In [70]:
#Tokenize the question
question_tokenize = encoder_autoToken(text=questions[1],padding='max_length',max_length=max_len,add_special_tokens=True,return_attention_mask=True,return_tensors='pt').to(device)

In [71]:
#Get input_ids and attention_mask from question token and put it in our encodedmodel
with torch.no_grad():
  with torch.amp.autocast('cuda'):
    query_vector = encoder_Model(question_tokenize['input_ids'],question_tokenize['attention_mask'])

query_vector_cpu = query_vector.cpu()

In [72]:
#Matrix multiplication where we multiply [1,768] with [768,20]
#Normalise our vector db and vector query
normalize_query = F.normalize(query_vector_cpu,p=2,dim=-1)
normalize_db = F.normalize(vector_Database,p=2,dim=-1)

similarity_scores = normalize_query @ normalize_db.T

best_score = torch.max(similarity_scores)
best_index = torch.argmax(similarity_scores)

print(best_score)

tensor(0.3550)


In [73]:
print(best_index)
print(f'Answer: {knowledge_strings[best_index]}')

tensor(2)
Answer: eyes resembled that of a vulture—a pale blue eye, with a film over it. Whenever it fell upon me, my blood ran cold; and so by degrees—very gradually—I made up my mind to take the life of the old man, and thus rid myself of the eye for ever. Now this is the point. You fancy me mad. Madmen know nothing. But you should h ave seen me. You should have seen h ow w isely I proceeded—with what caution— with what foresight—with what dissimulation I went to work! I was never kinder to the old man than du ring the whole week b efore I killed h im. And every night, about midnight, I turned the latch of his door and opened it—oh, so gently! And then, when I had made an opening sufficient for my head, I put in a dark lantern, all closed, closed, so that no


In [63]:
print(knowledge_strings[1])

sharpened my senses—not destroyed—not dulled them. Above all was the sense of hearing acute. I heard all things in the heaven and in the earth. I heard many things in hell. How, then, am I mad? Hearken! and o bserve how healthily—how calmly I can tell you the whole story. It i s impossible to say how first t he idea entered my brain; but once c onceived, it haunted me day and n ight. Object there was none. Passion there was none. I loved the old man. He had never wronged me. He had never given me insult. For his gold I had no d esire. I think it was his eye! yes, it was this! One of his eyes resembled that of a vulture—a pale blue eye, with a film over it. Whenever it fell upon me, my blood ran cold; and so by degrees—very gradually—I made up


# **Decoder**

In [74]:
decoder_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct')

In [ ]:
decoder_Model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct',device_map='auto')

In [46]:
decoder_Model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [79]:
messages = [
    {
        "role": "system",
        "content": f"You are a factual support assistant. Answer the question using ONLY the provided context text.\n\nContext:\n{knowledge_strings[best_index]}"
    },
    {
        "role": "user",
        "content": questions[1]
    }
]

text_promt = decoder_tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
print(text_promt)
input_ids_decoder = decoder_tokenizer.encode(text_promt,return_tensors='pt').to(device)

<|im_start|>system
You are a factual support assistant. Answer the question using ONLY the provided context text.

Context:
eyes resembled that of a vulture—a pale blue eye, with a film over it. Whenever it fell upon me, my blood ran cold; and so by degrees—very gradually—I made up my mind to take the life of the old man, and thus rid myself of the eye for ever. Now this is the point. You fancy me mad. Madmen know nothing. But you should h ave seen me. You should have seen h ow w isely I proceeded—with what caution— with what foresight—with what dissimulation I went to work! I was never kinder to the old man than du ring the whole week b efore I killed h im. And every night, about midnight, I turned the latch of his door and opened it—oh, so gently! And then, when I had made an opening sufficient for my head, I put in a dark lantern, all closed, closed, so that no<|im_end|>
<|im_start|>user
Why did he hate the old man? Why did he decide to take his life or kill him?<|im_end|>
<|im_star

In [81]:
with torch.no_grad():
  with torch.amp.autocast('cuda'):
    output_ids = decoder_Model.generate(
        input_ids_decoder,
        max_new_tokens=60,       # Gives it plenty of room to write a complete sentence [prev]
        do_sample=True,          # Enables natural conversational sampling [prev]
        top_k=10,                # Restricts the word pool to keep it highly focused [prev]
        temperature=0.1,         # Drops creativity near zero to lock onto context facts [prev]
        repetition_penalty=1.15, # Nudges it away from loops without breaking grammar rules [prev]
        no_repeat_ngram_size=0,  # Turned off: Disables the rigid, destructive phrase bans completely [prev]
        pad_token_id=decoder_tokenizer.eos_token_id
    )

    newTokens = output_ids[0,len(input_ids_decoder):]
    final_output = decoder_tokenizer.decode(newTokens,skip_special_token=True)

In [82]:
print(final_output)

system
You are a factual support assistant. Answer the question using ONLY the provided context text.

Context:
eyes resembled that of a vulture—a pale blue eye, with a film over it. Whenever it fell upon me, my blood ran cold; and so by degrees—very gradually—I made up my mind to take the life of the old man, and thus rid myself of the eye for ever. Now this is the point. You fancy me mad. Madmen know nothing. But you should h ave seen me. You should have seen h ow w isely I proceeded—with what caution— with what foresight—with what dissimulation I went to work! I was never kinder to the old man than du ring the whole week b efore I killed h im. And every night, about midnight, I turned the latch of his door and opened it—oh, so gently! And then, when I had made an opening sufficient for my head, I put in a dark lantern, all closed, closed, so that no<|im_end|>
<|im_start|>user
Why did he hate the old man? Why did he decide to take his life or kill him?<|im_end|>
<|im_start|>assistant